# 10주차 — Control II + Allocation: 호버 폐루프 완성

```
[Sensors] → [Estimation] → [Navigation] → [Control] → [Allocation] → [Plant]
                                                        ▲ 이번 주
```

**드론이 처음 뜨는 주.** 캐스케이드(위치→자세→각속도)와 믹서를 잇는다.

## 이론 요약
- 캐스케이드가 계층인 이유: 바깥 루프는 안쪽 루프의 setpoint 를 만든다
  (위치 오차 → 가속도 → **자세**·추력 → 각속도 → 토크)
- 쿼터니언 자세오차(개념): '지금 자세에서 목표 자세로 가는 최단 회전'
- Allocation(믹서): 정규화 토크/추력 → 모터 4개. **제어기는 기체를 모른다** —
  기체 차이는 믹서가 흡수 (PX4 구조. 14주차에 이 말의 의미를 확인한다)
- Mixer 블록의 frame 파라미터: quad_x/quad_plus/hexa_x/octo_x

In [ ]:
# 뷰어 열기 — 뼈대 모델을 브라우저에서 편집한다.
# 완성하면 뷰어의 '저장' → week10_model.json 이 다운로드된다.
# 그 파일을 이 폴더의 같은 이름 파일에 **덮어쓰면 제출 완료** (채점 셀로 확인).
import sys, webbrowser; sys.path.insert(0, '../..')
from loopsym import load_json, render_html
p = render_html(load_json('week10_model.json'), 'week10_viewer.html',
                title='10주차', t_end=8.0, dt=0.004,
                save_name='week10_model.json')
webbrowser.open('file://' + p)

## 실습 — 캐스케이드 잇기
뼈대에서 빠진 것: ①pos_ctrl 출력(roll/pitch/yaw_cmd)→att_ctrl 의 att_cmd,
②att_ctrl 출력(rate_sp)→rate_ctrl, ③pos_ctrl 의 thrust→mixer.
포트에 마우스를 올리면 이름이 뜬다. 완성하면: 북 2m + 고도 6m 이동 후 정지 호버.

In [ ]:
# 노트북에서 바로 실행해 보기 (뷰어 없이) — 저장한 모델을 로드해 시뮬
import sys; sys.path.insert(0, '../..')
from loopsym import load_json, Simulator
bd = load_json('week10_model.json')
sim = Simulator(bd); sim.run(8.0, 0.004)
import matplotlib.pyplot as plt
for b in sim.bd.blocks:
    if type(b).__name__ == 'Scope':
        plt.figure(figsize=(8,2.6)); plt.title(b.name); plt.grid(True)
        for i, lab in enumerate(b.labels): plt.plot(b.t, b.data[i], label=lab)
        plt.legend()

## 과제 (팀 프로젝트 시작)
1. week10_model.json 완성 (채점기: 최종 NED=[2,0,−6] ±0.1m + 정착)
2. north_sp 의 Step 높이를 5m 로 키우고 오버슈트를 관찰 — 어느 게인을
   만지면 줄어드는가? (튜닝 근거를 남겨라 — 15주차 발표 재료)

In [ ]:
# ✅ 채점 — 전부 초록색이 되면 이번 주 완료!
!python -m pytest test_week10.py -q